# Standard problem 3

## Problem specification

This problem is to calculate a single domain limit of a cubic magnetic particle. This is the size $L$ of equal energy for the so-called flower state (which one may also call a splayed state or a modified single-domain state) on the one hand, and the vortex or curling state on the other hand.

Geometry:

A cube with edge length, $L$, expressed in units of the intrinsic length scale, $l_\text{ex} = \sqrt{A/K_\text{m}}$, where $K_\text{m}$ is a magnetostatic energy density, $K_\text{m} = \frac{1}{2}\mu_{0}M_\text{s}^{2}$.

Material parameters: 

- uniaxial anisotropy $K_\text{u}$ with $K_\text{u} = 0.1 K_\text{m}$, and with the easy axis directed parallel to a principal axis of the cube (0, 0, 1),
- exchange energy constant is $A = \frac{1}{2}\mu_{0}M_\text{s}^{2}l_\text{ex}^{2}$.

More details about the standard problem 3 can be found in Ref. 1.

## Simulation

Firstly, we import all necessary modules.

In [ ]:
import numpy as np

import neuralmag as nm

mu0 = 4 * np.pi * 1e-7

The following two functions are used for initialising the system's magnetisation [1].

In [ ]:
import numpy as np


# Function for initiaising the flower state.
def m_init_flower(state):
    return nm.VectorFunction(state).fill((0.0, 0.0, 1))


# Function for initialising the vortex state.
def m_init_vortex(state):
    x, y, z = state.coordinates(spaces=("n", "n", "n"))
    out = np.stack([np.ones_like(x) * 5e-9, z - 50e-9, 50e-9 - y], axis=3)
    norm = np.linalg.norm(out, axis=-1, keepdims=True)
    return nm.VectorFunction(state, tensor=state.tensor(out / norm))

The following function is used for convenience. It takes two arguments:

- $L$ - the cube edge length in units of $l_\text{ex}$, and
- the function for initialising the system's magnetisation.

It returns the relaxed system object.

Please refer to other tutorials for more details on how to create system objects and drive them using specific drivers.

In [ ]:
def minimise_system_energy(L, m_init):
    print("L={:7}, {} ".format(L, m_init.__name__), end="")
    N = 16  # discretisation in one dimension
    cubesize = 100e-9  # cube edge length (m)
    cellsize = cubesize / N  # discretisation in all three dimensions.
    lex = cubesize / L  # exchange length.

    Km = 1e6  # magnetostatic energy density (J/m**3)
    Ms = np.sqrt(2 * Km / mu0)  # magnetisation saturation (A/m)
    A = 0.5 * mu0 * Ms**2 * lex**2  # exchange energy constant
    K = 0.1 * Km  # Uniaxial anisotropy constant
    u = (0, 0, 1)  # Uniaxial anisotropy easy-axis

    mesh = nm.Mesh([N, N, N], [cellsize, cellsize, cellsize], [0.0, 0.0, 0.0])
    state = nm.State(mesh)

    state.material.Ms = Ms
    state.material.A = A
    state.material.alpha = 0.5
    state.material.Ku = 0.1 * Km
    state.material.Ku_axis = [0, 0, 1]

    state.m = m_init(state)

    nm.ExchangeField().register(state, "exchange")
    nm.UniaxialAnisotropyField().register(state, "aniso")
    nm.DemagField().register(state, "demag")
    nm.TotalField("exchange", "demag", "aniso").register(state)

    # relax the system
    llg = nm.LLGSolver(state)
    llg.step(20e-10)
    return state

### Relaxed magnetisation states

Now, we show the magnetisation configurations of two relaxed states.

**Vortex** state:

In [ ]:
# NBVAL_IGNORE_OUTPUT
state = minimise_system_energy(8, m_init_vortex)

**Flower** state:

In [ ]:
# NBVAL_IGNORE_OUTPUT
state = minimise_system_energy(8, m_init_flower)

### Energy crossing

We can plot the energies of both vortex and flower states as a function of cube edge length $L$. This will give us an idea where the state transition occurrs. We can achieve that by simply looping over the edge lengths $L$ of interest, computing the energy of both vortex and flower states, and finally, plotting the energy dependence.

In [ ]:
# NBVAL_IGNORE_OUTPUT
L_array = np.linspace(8, 9, 5)
vortex_energies, flower_energies = [], []
for L in L_array:
    vortex = minimise_system_energy(L, m_init_vortex)
    flower = minimise_system_energy(L, m_init_flower)
    vortex_energies.append(vortex.E)
    flower_energies.append(flower.E)

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(L_array, vortex_energies, "o-", label="vortex")
plt.plot(L_array, flower_energies, "o-", label="flower")
plt.xlabel("L (lex)")
plt.ylabel("E (J)")
plt.grid()
plt.legend();

From the plot, we can see that the energy crossing occurrs between $8.4l_\text{ex}$ and $8.6l_\text{ex}$, so we can employ a root-finding (e.g. bisection) algorithm to find the exact crossing.

In [ ]:
# NBVAL_IGNORE_OUTPUT
from scipy.optimize import bisect


def energy_difference(L):
    vortex = minimise_system_energy(L, m_init_vortex)
    flower = minimise_system_energy(L, m_init_flower)
    return vortex.E - flower.E


cross_section = bisect(energy_difference, 8.4, 8.6, xtol=0.02)

print(f"\nThe energy crossing occurs at {cross_section}*lex")

## References

[1] µMAG Site Directory http://www.ctcms.nist.gov/~rdm/mumag.org.

This tutorial was original created by https://ubermag.github.io/